# Model quality review

Review diagnostics, unnamed elements, repeated names, and empty definitions.

This sample uses only standard SysML v2 concepts and reads the model through `memo_model.py`, compiled by the project's selected compiler.

In [ ]:
import sys
from pathlib import Path
from collections import Counter, defaultdict

# memo_model.py sits in the project's analysis/ folder, above this notebook.
sys.path.insert(0, str(next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / 'memo_model.py').is_file())))
import memo_model

# Compiled by the project's selected compiler (toolchain.compiler), as every
# MEMO command is: change the setting and this notebook follows.
model = memo_model.load()
diagnostics = model.diagnostics
SYSML_FILES = model.documents
print(f'Loaded {len(SYSML_FILES)} SysML files, compiled by {model.compiler} {model.compiler_version or ""}')

In [ ]:
from IPython.display import display

usages = model.elements('Usage', include_subtypes=True)
definitions = model.elements('Definition', include_subtypes=True)
semantic_elements = usages + definitions

unnamed = [item for item in semantic_elements if not (item.name or item.declared_name)]
by_name = defaultdict(list)
for item in semantic_elements:
    value = item.qualified_name or item.name or item.declared_name
    name = str(value) if value else None
    if name:
        by_name[name].append(item)
duplicates = [(name, len(items), sorted({item.type for item in items})) for name, items in by_name.items() if len(items) > 1]
empty_definitions = [
    item for item in definitions
    if not any(child.is_a('Usage', 'Definition') for child in item.owned_elements)
]

display({
    'parser/semantic errors': len(diagnostics.errors),
    'warnings': len(diagnostics.warnings),
    'unnamed usages or definitions': len(unnamed),
    'repeated qualified names': len(duplicates),
    'definitions with no owned usages/definitions': len(empty_definitions),
})

print('Repeated qualified names (first 30)')
display(sorted(duplicates, key=lambda item: item[1], reverse=True)[:30])

print('Empty definition candidates (first 30)')
[(item.type, item.qualified_name or item.name) for item in empty_definitions[:30]]

In [ ]:
print('First diagnostics to investigate')
for diagnostic in diagnostics.errors[:20] + diagnostics.warnings[:20]:
    print(diagnostic)